# IMPORT THƯ VIỆN CƠ BẢN

In [ ]:
# Import các thư viện cơ bản
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Thiết lập style cho matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Import NLTK
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.stem import WordNetLemmatizer

# Download NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

# Import spaCy
import spacy
nlp = spacy.load('en_core_web_sm')

# Import scikit-learn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split

# Import Gensim for Word2Vec
from gensim.models import Word2Vec

# Import WordCloud
from wordcloud import WordCloud

# Import re cho regex
import re
from collections import Counter
import time

from transformers import BertTokenizer

# IMPORT DATA

In [ ]:
df = pd.read_csv('../data/text/IMDB Dataset.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df_sample = pd.concat([
    df[df['sentiment'] == 'positive'].sample(2500, random_state=42),
    df[df['sentiment'] == 'negative'].sample(2500, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Số lượng mẫu: {len(df_sample):,}")


df_work = df_sample.copy()

Lấy 10%  mỗi nhãn để chạy

In [ ]:
df_work.info()

# TOKENIZATION

## Word Tokenization

In [ ]:
sample_review = df_work['review'].iloc[0]
print("Review gốc (1 review):")
print(sample_review[:500])

In [ ]:
# Tokenization với NLTK
print("\n" + "=" * 80)
print("WORD TOKENIZATION (NLTK):")
print("=" * 80)
tokens_nltk = word_tokenize(sample_review)
print(f"Số tokens: {len(tokens_nltk)}")
print(f"\n50 tokens đầu tiên:\n{tokens_nltk[:50]}")

# Tokenization với spaCy
print("\n" + "=" * 80)
print("WORD TOKENIZATION (spaCy):")
print("=" * 80)
doc_spacy = nlp(sample_review)
tokens_spacy = [token.text for token in doc_spacy]
print(f"Số tokens: {len(tokens_spacy)}")
print(f"\n50 tokens đầu tiên:\n{tokens_spacy[:50]}")

## Sentence Tokenization

In [ ]:
# Sentence Tokenization với NLTK
print("=" * 80)
print("SENTENCE TOKENIZATION (NLTK):")
print("=" * 80)

sentences_nltk = sent_tokenize(sample_review)
print(f"Số câu: {len(sentences_nltk)}")
print("\n5 câu đầu tiên:")
for i, sent in enumerate(sentences_nltk[:5], 1):
    print(f"\n[{i}] {sent}")

In [ ]:
# Sentence Tokenization với spaCy
print("\n" + "=" * 80)
print("SENTENCE TOKENIZATION (spaCy):")
print("=" * 80)

doc_spacy = nlp(sample_review)
sentences_spacy = [sent.text for sent in doc_spacy.sents]
print(f"Số câu: {len(sentences_spacy)}")
print("\n5 câu đầu tiên:")
for i, sent in enumerate(sentences_spacy[:5], 1):
    print(f"\n[{i}] {sent}")

## Subword Tokenization

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
tokens_subword = tokenizer.tokenize(sample_review)

print(f"Văn bản gốc: {sample_review}")
print(f"Kết quả Subword: {tokens_subword}")

## Compare & Discuss

In [ ]:
# So sánh thời gian xử lý trên 1000 reviews
test_reviews = df_work['review'].head(1000).tolist()

print("SO SÁNH HIỆU SUẤT TOKENIZATION (1000 reviews):")
print("=" * 80)

# NLTK word tokenization
start = time.time()
for review in test_reviews:
    tokens = word_tokenize(review)
time_nltk = time.time() - start
print(f"NLTK word_tokenize: {time_nltk:.3f} giây")

# spaCy tokenization
start = time.time()
for review in test_reviews:
    doc = nlp(review)
    tokens = [token.text for token in doc]
time_spacy = time.time() - start
print(f"spaCy tokenization: {time_spacy:.3f} giây")

# Simple split
start = time.time()
for review in test_reviews:
    tokens = review.split()
time_simple = time.time() - start
print(f"Simple split: {time_simple:.3f} giây")

# Vẽ biểu đồ so sánh
methods = ['NLTK', 'spaCy', 'Simple Split']
times = [time_nltk, time_spacy, time_simple]

plt.figure(figsize=(10, 6))
bars = plt.bar(methods, times, color=['#3498db', '#e74c3c', '#2ecc71'], edgecolor='black', alpha=0.7)
plt.title('So sánh thời gian Tokenization (1000 reviews)', fontsize=14, fontweight='bold')
plt.ylabel('Thời gian (giây)', fontsize=12)
plt.xlabel('Phương pháp', fontsize=12)

# Thêm giá trị lên cột
for i, bar in enumerate(bars):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{times[i]:.3f}s',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Tokenize toàn bộ dataset với NLTK
start = time.time()

df_work['tokens'] = df_work['review'].apply(word_tokenize)
df_work['token_count'] = df_work['tokens'].apply(len)

elapsed = time.time() - start
print(f"Hoàn thành trong {elapsed:.2f} giây")
print(f"\nSố token trung bình mỗi review: {df_work['token_count'].mean():.1f}")
print(f"Token ít nhất: {df_work['token_count'].min()}")
print(f"Token nhiều nhất: {df_work['token_count'].max()}")

# REMOVING STOP WORDS

Thống kê danh sách Stop Words

In [ ]:
stop_words = set(stopwords.words('english'))

print(f"Số lượng stop words trong NLTK: {len(stop_words)}")
print("\n50 stop words đầu tiên:")
print(list(stop_words)[:50])

Phân tích tần suất các Stop Words trong Dataset

In [ ]:
# Đếm tần suất các từ trong dataset
all_tokens = []
for tokens in df_work['tokens']:
    all_tokens.extend([token.lower() for token in tokens if token.isalpha()])

word_freq = Counter(all_tokens)
total_words = len(all_tokens)
unique_words = len(word_freq)

print("THỐNG KÊ TỪ VỰNG:")
print("=" * 80)
print(f"Tổng số từ (tokens): {total_words:,}")
print(f"Số từ unique: {unique_words:,}")
print(f"\n10 từ xuất hiện nhiều nhất:")
for word, count in word_freq.most_common(10):
    percentage = (count / total_words) * 100
    is_stopword = "[STOP]" if word in stop_words else ""
    print(f"  {word:15s}: {count:6,} lần ({percentage:5.2f}%) {is_stopword}")

In [ ]:
# Đếm số stop words và non-stop words
stopwords_count = sum(count for word, count in word_freq.items() if word in stop_words)
non_stopwords_count = total_words - stopwords_count

print("\n" + "=" * 80)
print("PHÂN TÍCH STOP WORDS:")
print("=" * 80)
print(f"Số lượng stop words: {stopwords_count:,} ({stopwords_count/total_words*100:.2f}%)")
print(f"Số lượng non-stop words: {non_stopwords_count:,} ({non_stopwords_count/total_words*100:.2f}%)")

# Vẽ biểu đồ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
sizes = [stopwords_count, non_stopwords_count]
labels = ['Stop Words', 'Non-Stop Words']
colors = ['#e74c3c', '#2ecc71']
explode = (0.05, 0)

axes[0].pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
            shadow=True, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[0].set_title('Tỷ lệ Stop Words vs Non-Stop Words', fontsize=14, fontweight='bold')

# Bar chart - Top 20 từ
top_20 = word_freq.most_common(20)
words = [w[0] for w in top_20]
counts = [w[1] for w in top_20]
colors_bar = ['#e74c3c' if w in stop_words else '#3498db' for w in words]

axes[1].barh(range(len(words)), counts, color=colors_bar, edgecolor='black', alpha=0.7)
axes[1].set_yticks(range(len(words)))
axes[1].set_yticklabels(words)
axes[1].invert_yaxis()
axes[1].set_xlabel('Tần suất', fontsize=12)
axes[1].set_title('20 từ xuất hiện nhiều nhất\n(Đỏ: Stop words, Xanh: Non-stop words)', 
                  fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()


Loại bỏ Stop Words

In [ ]:
def remove_stopwords(tokens, stop_words):
    return [token for token in tokens if token.lower() not in stop_words and token.isalpha()]

start = time.time()

df_work['tokens_no_stopwords'] = df_work['tokens'].apply(lambda x: remove_stopwords(x, stop_words))
df_work['token_count_no_stopwords'] = df_work['tokens_no_stopwords'].apply(len)

elapsed = time.time() - start
print(f"Hoàn thành trong {elapsed:.2f} giây")

Thống kê sau khi loại bỏ stop words

In [ ]:
# Thống kê sau khi loại stop words
all_tokens_no_sw = []
for tokens in df_work['tokens_no_stopwords']:
    all_tokens_no_sw.extend([token.lower() for token in tokens])

word_freq_no_sw = Counter(all_tokens_no_sw)
total_words_no_sw = len(all_tokens_no_sw)
unique_words_no_sw = len(word_freq_no_sw)

print("SO SÁNH TRƯỚC VÀ SAU KHI LOẠI STOP WORDS:")
print("=" * 80)
print(f"{'Metric':<30} {'Trước':<20} {'Sau':<20} {'Thay đổi'}")
print("=" * 80)

# Tổng số từ
change_total = ((total_words_no_sw - total_words) / total_words) * 100
print(f"{'Tổng số từ':<30} {total_words:>18,} {total_words_no_sw:>18,} {change_total:>10.1f}%")

# Số từ unique
change_unique = ((unique_words_no_sw - unique_words) / unique_words) * 100
print(f"{'Số từ unique':<30} {unique_words:>18,} {unique_words_no_sw:>18,} {change_unique:>10.1f}%")

# Trung bình tokens/review
avg_before = df_work['token_count'].mean()
avg_after = df_work['token_count_no_stopwords'].mean()
change_avg = ((avg_after - avg_before) / avg_before) * 100
print(f"{'Trung bình tokens/review':<30} {avg_before:>18.1f} {avg_after:>18.1f} {change_avg:>10.1f}%")

print("=" * 80)

In [ ]:
# Vẽ biểu đồ so sánh
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. So sánh vocabulary size
categories = ['Tổng số từ', 'Từ unique']
before = [total_words, unique_words]
after = [total_words_no_sw, unique_words_no_sw]

x = np.arange(len(categories))
width = 0.35

bars1 = axes[0, 0].bar(x - width/2, before, width, label='Trước', color='#e74c3c', edgecolor='black', alpha=0.7)
bars2 = axes[0, 0].bar(x + width/2, after, width, label='Sau', color='#2ecc71', edgecolor='black', alpha=0.7)

axes[0, 0].set_xlabel('Metric', fontsize=12)
axes[0, 0].set_ylabel('Số lượng', fontsize=12)
axes[0, 0].set_title('So sánh Vocabulary Size', fontsize=14, fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(categories)
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Thêm giá trị lên cột
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                       f'{int(height):,}',
                       ha='center', va='bottom', fontsize=9)

# 2. Phân bố số tokens/review
axes[0, 1].hist([df_work['token_count'], df_work['token_count_no_stopwords']], 
                bins=50, label=['Trước', 'Sau'], color=['#e74c3c', '#2ecc71'], alpha=0.6, edgecolor='black')
axes[0, 1].set_xlabel('Số tokens', fontsize=12)
axes[0, 1].set_ylabel('Tần suất', fontsize=12)
axes[0, 1].set_title('Phân bố số tokens/review', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Top 20 từ trước khi loại stop words
top_20_before = word_freq.most_common(20)
words_before = [w[0] for w in top_20_before]
counts_before = [w[1] for w in top_20_before]

axes[1, 0].barh(range(len(words_before)), counts_before, color='#e74c3c', edgecolor='black', alpha=0.7)
axes[1, 0].set_yticks(range(len(words_before)))
axes[1, 0].set_yticklabels(words_before)
axes[1, 0].invert_yaxis()
axes[1, 0].set_xlabel('Tần suất', fontsize=12)
axes[1, 0].set_title('Top 20 từ - TRƯỚC loại stop words', fontsize=12, fontweight='bold')
axes[1, 0].grid(axis='x', alpha=0.3)

# 4. Top 20 từ sau khi loại stop words
top_20_after = word_freq_no_sw.most_common(20)
words_after = [w[0] for w in top_20_after]
counts_after = [w[1] for w in top_20_after]

axes[1, 1].barh(range(len(words_after)), counts_after, color='#2ecc71', edgecolor='black', alpha=0.7)
axes[1, 1].set_yticks(range(len(words_after)))
axes[1, 1].set_yticklabels(words_after)
axes[1, 1].invert_yaxis()
axes[1, 1].set_xlabel('Tần suất', fontsize=12)
axes[1, 1].set_title('Top 20 từ - SAU loại stop words', fontsize=12, fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

Word Cloud

In [ ]:
# Tạo WordCloud
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# WordCloud trước khi loại stop words
text_before = ' '.join(all_tokens)
wordcloud_before = WordCloud(width=800, height=400, background_color='white', 
                             colormap='Reds', max_words=100).generate(text_before)
axes[0].imshow(wordcloud_before, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('WordCloud TRƯỚC loại stop words', fontsize=14, fontweight='bold')

# WordCloud sau khi loại stop words
text_after = ' '.join(all_tokens_no_sw)
wordcloud_after = WordCloud(width=800, height=400, background_color='white', 
                            colormap='Greens', max_words=100).generate(text_after)
axes[1].imshow(wordcloud_after, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('WordCloud SAU loại stop words', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# STEMMING AND LEMMATIZATION

In [ ]:
example_words = ['running', 'runs', 'ran', 'runner', 'easily', 'fairly', 'playing', 
                 'played', 'player', 'better', 'best', 'good', 'movies', 'watched', 'watching']

## Stemming

Stemming với Porter Stemmer

In [ ]:
porter = PorterStemmer()
print("PORTER STEMMER:")
print("=" * 80)
print(f"{'Từ gốc':<20} → {'Stem'}")
print("=" * 80)
for word in example_words:
    stem = porter.stem(word)
    print(f"{word:<20} → {stem}")

Stemming với Snowball Stemmer

In [ ]:
snowball = SnowballStemmer('english')

print("\SNOWBALL STEMMER:")
print("=" * 80)
print(f"{'Từ gốc':<20} → {'Stem'}")
print("=" * 80)
for word in example_words:
    stem = snowball.stem(word)
    print(f"{word:<20} → {stem}")

## Lemmatization

In [ ]:
lemmatizer = WordNetLemmatizer()

print("\nLEMMATIZATION:")
print("=" * 80)
print(f"{'Từ gốc':<20} → {'Lemma'}")
print("=" * 80)
for word in example_words:
    lemma = lemmatizer.lemmatize(word, pos='v')  # pos='v' cho động từ
    print(f"{word:<20} → {lemma}")

## So sánh Stemming và Lemmtization

In [ ]:
print("\nSO SÁNH:")
print("=" * 100)
print(f"{'Từ gốc':<20} {'Porter':<20} {'Snowball':<20} {'Lemmatization'}")
print("=" * 100)

for word in example_words:
    porter_stem = porter.stem(word)
    snowball_stem = snowball.stem(word)
    lemma = lemmatizer.lemmatize(word, pos='v')
    print(f"{word:<20} {porter_stem:<20} {snowball_stem:<20} {lemma}")

## Apply lên dataset + Đo hiệu suất và kết quả

In [ ]:
def apply_stemming(tokens, stemmer):
    return [stemmer.stem(token.lower()) for token in tokens if token.isalpha()]

def apply_lemmatization(tokens):
    return [lemmatizer.lemmatize(token.lower(), pos='v') for token in tokens if token.isalpha()]

# Áp dụng Porter Stemmer
print("Porter Stemmer")
start = time.time()
df_work['tokens_porter'] = df_work['tokens_no_stopwords'].apply(lambda x: apply_stemming(x, porter))
time_porter = time.time() - start
print(f"Hoàn thành trong {time_porter:.2f} giây")

# Áp dụng Snowball Stemmer
print("\nSnowball Stemmer")
start = time.time()
df_work['tokens_snowball'] = df_work['tokens_no_stopwords'].apply(lambda x: apply_stemming(x, snowball))
time_snowball = time.time() - start
print(f"Hoàn thành trong {time_snowball:.2f} giây")

# Áp dụng Lemmatization
print("\nLemmatization")
start = time.time()
df_work['tokens_lemma'] = df_work['tokens_no_stopwords'].apply(apply_lemmatization)
time_lemma = time.time() - start
print(f"Hoàn thành trong {time_lemma:.2f} giây")

In [ ]:
# Tính toán vocabulary size
vocab_original = len(set([token.lower() for tokens in df_work['tokens'] for token in tokens if token.isalpha()]))
vocab_porter = len(set([token for tokens in df_work['tokens_porter'] for token in tokens]))
vocab_snowball = len(set([token for tokens in df_work['tokens_snowball'] for token in tokens]))
vocab_lemma = len(set([token for tokens in df_work['tokens_lemma'] for token in tokens]))

print("\nSO SÁNH KẾT QUẢ:")
print("=" * 100)
print(f"{'Phương pháp':<25} {'Vocabulary Size':<20} {'Thời gian (s)':<20} {'Giảm (%)'}")
print("=" * 100)
print(f"{'Original (không xử lý)':<25} {vocab_original:<20,} {'-':<20} {'-'}")
print(f"{'Porter Stemmer':<25} {vocab_porter:<20,} {time_porter:<20.2f} {(1-vocab_porter/vocab_original)*100:>10.1f}%")
print(f"{'Snowball Stemmer':<25} {vocab_snowball:<20,} {time_snowball:<20.2f} {(1-vocab_snowball/vocab_original)*100:>10.1f}%")
print(f"{'Lemmatization':<25} {vocab_lemma:<20,} {time_lemma:<20.2f} {(1-vocab_lemma/vocab_original)*100:>10.1f}%")
print("=" * 100)

In [ ]:
# Vẽ biểu đồ so sánh
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. So sánh Vocabulary Size
methods = ['Original', 'Porter', 'Snowball', 'Lemma']
vocab_sizes = [vocab_original, vocab_porter, vocab_snowball, vocab_lemma]
colors = ['#95a5a6', '#e74c3c', '#e67e22', '#2ecc71']

bars = axes[0].bar(methods, vocab_sizes, color=colors, edgecolor='black', alpha=0.7)
axes[0].set_title('So sánh Vocabulary Size', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Số từ unique', fontsize=12)
axes[0].grid(axis='y', alpha=0.3)

for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{vocab_sizes[i]:,}',
                ha='center', va='bottom', fontweight='bold')

# 2. So sánh thời gian xử lý
times = [time_porter, time_snowball, time_lemma]
methods_time = ['Porter', 'Snowball', 'Lemma']
colors_time = ['#e74c3c', '#e67e22', '#2ecc71']

bars = axes[1].bar(methods_time, times, color=colors_time, edgecolor='black', alpha=0.7)
axes[1].set_title('So sánh thời gian xử lý', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Thời gian (giây)', fontsize=12)
axes[1].grid(axis='y', alpha=0.3)

for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{times[i]:.2f}s',
                ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# TEXT VECTORIZATION

In [ ]:
df_work['processed_text'] = df_work['tokens_lemma'].apply(lambda x: ' '.join(x))

print("VÍ DỤ TEXT SAU TIỀN XỬ LÝ:")
print("=" * 80)
for i in range(3):
    print(f"\n[{i+1}] Original: {df_work['review'].iloc[i][:150]}...")
    print(f"    Processed: {df_work['processed_text'].iloc[i][:150]}...")

Khởi tạo chung

In [ ]:
bow_vectorizer = CountVectorizer(
    max_features=5000,        # Giới hạn 5000 từ phổ biến nhất
    min_df=5,                 # Từ phải xuất hiện ít nhất 5 documents
    max_df=0.8,               # Từ xuất hiện nhiều hơn 80% docs sẽ bị loại
    ngram_range=(1, 2)        # Unigrams và bigrams
)

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2),
    sublinear_tf=True        # Sử dụng log scaling cho TF
)


## Bag-of-Words (BoW)

In [ ]:
start = time.time()
bow_matrix = bow_vectorizer.fit_transform(df_work['processed_text'])
time_bow = time.time() - start

print(f"  Hoàn thành trong {time_bow:.2f} giây")
print(f"\nThông tin BoW matrix:")
print(f"  Shape: {bow_matrix.shape}")
print(f"  Số documents: {bow_matrix.shape[0]:,}")
print(f"  Vocabulary size: {bow_matrix.shape[1]:,}")
print(f"  Sparsity: {(1 - bow_matrix.nnz / (bow_matrix.shape[0] * bow_matrix.shape[1])) * 100:.2f}%")
print(f"  Memory size: {bow_matrix.data.nbytes / 1024**2:.2f} MB")

In [ ]:
# vocabulary
bow_vocab = bow_vectorizer.get_feature_names_out()
print(f"\n50 từ đầu tiên trong vocabulary:")
print(bow_vocab[:50])

In [ ]:
# vector của 1 document
doc_idx = 0
doc_vector = bow_matrix[doc_idx].toarray()[0]
non_zero_indices = np.nonzero(doc_vector)[0]

print(f"\nVector của document {doc_idx}:")
print(f"  Số từ khác 0: {len(non_zero_indices)}")
print(f"  Top 20 từ có tần suất cao:")
top_indices = doc_vector.argsort()[-20:][::-1]
for idx in top_indices:
    if doc_vector[idx] > 0:
        print(f"    {bow_vocab[idx]:20s}: {int(doc_vector[idx])}")

## TF-IDF Vectorization

In [ ]:
start = time.time()
tfidf_matrix = tfidf_vectorizer.fit_transform(df_work['processed_text'])
time_tfidf = time.time() - start

print(f"  Hoàn thành trong {time_tfidf:.2f} giây")
print(f"\nThông tin TF-IDF matrix:")
print(f"  Shape: {tfidf_matrix.shape}")
print(f"  Số documents: {tfidf_matrix.shape[0]:,}")
print(f"  Vocabulary size: {tfidf_matrix.shape[1]:,}")
print(f"  Sparsity: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")
print(f"  Memory size: {tfidf_matrix.data.nbytes / 1024**2:.2f} MB")

In [ ]:
# Xem vector của 1 document với TF-IDF
doc_idx = 0
tfidf_vector = tfidf_matrix[doc_idx].toarray()[0]
tfidf_vocab = tfidf_vectorizer.get_feature_names_out()

print(f"\nTF-IDF vector của document {doc_idx}:")
print(f"  Top 20 từ có TF-IDF cao nhất:")
top_indices = tfidf_vector.argsort()[-20:][::-1]
for idx in top_indices:
    if tfidf_vector[idx] > 0:
        print(f"    {tfidf_vocab[idx]:20s}: {tfidf_vector[idx]:.4f}")

## So sánh BoW và TF-IDF

In [ ]:
# So sánh trên cùng 1 document
doc_idx = 5

# BoW
bow_vector = bow_matrix[doc_idx].toarray()[0]
bow_top_20 = bow_vector.argsort()[-20:][::-1]

# TF-IDF
tfidf_vector = tfidf_matrix[doc_idx].toarray()[0]
tfidf_top_20 = tfidf_vector.argsort()[-20:][::-1]

print("SO SÁNH BoW vs TF-IDF trên cùng 1 document:")
print("=" * 100)
print(f"{'Top 20 BoW':<50} {'Top 20 TF-IDF'}")
print("=" * 100)

for i in range(20):
    bow_word = bow_vocab[bow_top_20[i]]
    bow_val = bow_vector[bow_top_20[i]]
    
    tfidf_word = tfidf_vocab[tfidf_top_20[i]]
    tfidf_val = tfidf_vector[tfidf_top_20[i]]
    
    if bow_val > 0 or tfidf_val > 0:
        print(f"{bow_word:30s} ({bow_val:4.0f})   {tfidf_word:30s} ({tfidf_val:6.4f})")

In [ ]:
# Vẽ biểu đồ so sánh phân phối giá trị
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# BoW distribution
bow_nonzero = bow_matrix.data
axes[0].hist(bow_nonzero, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].set_title('Phân phối giá trị BoW', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Giá trị (tần suất)', fontsize=12)
axes[0].set_ylabel('Số lượng', fontsize=12)
axes[0].set_yscale('log')
axes[0].grid(alpha=0.3)

# TF-IDF distribution
tfidf_nonzero = tfidf_matrix.data
axes[1].hist(tfidf_nonzero, bins=50, color='#e74c3c', edgecolor='black', alpha=0.7)
axes[1].set_title('Phân phối giá trị TF-IDF', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Giá trị (TF-IDF score)', fontsize=12)
axes[1].set_ylabel('Số lượng', fontsize=12)
axes[1].set_yscale('log')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Word2Vec Embeddings

In [ ]:
# Chuẩn bị dữ liệu cho Word2Vec (cần danh sách các danh sách tokens)
sentences = df_work['tokens_lemma'].tolist()

# Train Word2Vec model
print("Đang train Word2Vec model...")
start = time.time()

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,      # Số chiều của vector
    window=5,             # Context window
    min_count=5,          # Bỏ qua từ xuất hiện < 5 lần
    workers=4,            # Số threads
    sg=0,                 # 0=CBOW, 1=Skip-gram
    epochs=10
)

time_w2v = time.time() - start
print(f"  Hoàn thành trong {time_w2v:.2f} giây")
print(f"\nThông tin Word2Vec model:")
print(f"  Vocabulary size: {len(w2v_model.wv):,}")
print(f"  Vector dimensions: {w2v_model.wv.vector_size}")

In [ ]:
# Xem vector của 1 từ
word = 'movie'
if word in w2v_model.wv:
    vector = w2v_model.wv[word]
    print(f"\nVector của từ '{word}':")
    print(f"  Shape: {vector.shape}")
    print(f"  10 giá trị đầu tiên: {vector[:10]}")

In [ ]:
# Tìm từ tương tự
test_words = ['good', 'bad', 'movie', 'actor', 'love', 'hate']

print("\nTÌM TỪ TƯƠNG TỰ:")
print("=" * 80)
for word in test_words:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=5)
        print(f"\n'{word}' tương tự với:")
        for similar_word, score in similar:
            print(f"  - {similar_word:<15s} (similarity: {score:.4f})")

In [ ]:
def vector_calculate(positive_words, negative_words):
    try:
        result = w2v_model.wv.most_similar(
            positive=positive_words, 
            negative=negative_words, 
            topn=5
        )
        
        pos_str = " + ".join(positive_words)
        neg_str = " - ".join(negative_words)
        
        if neg_str:
            formula = f"{pos_str} - ({neg_str})"
        else:
            formula = pos_str
            
        print(f"\nPHÉP TOÁN VECTOR: {formula} = ?")
        print("=" * 80)
        
        for word, score in result:
            print(f"  {word:<20s} (similarity: {score:.4f})")
            
    except KeyError as e:
        print(f"Lỗi: Từ '{e.args[0]}' không có trong bộ từ điển (vocabulary).")

vector_calculate(positive_words=['king', 'woman'], negative_words=['man'])
vector_calculate(positive_words=['good', 'excellent'], negative_words=['bad'])
vector_calculate(positive_words=['movie', 'music'], negative_words=[])

In [ ]:
def get_doc_vector(tokens, model):
    """
    Tạo document vector bằng cách lấy trung bình các word vectors
    """
    vectors = []
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
    
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.wv.vector_size)

# Tạo document vectors cho toàn bộ dataset
print("Đang tạo document vectors...")
start = time.time()
doc_vectors = np.array([get_doc_vector(tokens, w2v_model) for tokens in df_work['tokens_lemma']])
time_doc_vec = time.time() - start

print(f"  Hoàn thành trong {time_doc_vec:.2f} giây")
print(f"\nDocument vectors shape: {doc_vectors.shape}")
print(f"Memory size: {doc_vectors.nbytes / 1024**2:.2f} MB")

## So sánh tổng quan các phương pháp

In [ ]:
# Tổng hợp thông tin
comparison_data = {
    'Phương pháp': ['Bag-of-Words', 'TF-IDF', 'Word2Vec'],
    'Dimensions': [bow_matrix.shape[1], tfidf_matrix.shape[1], doc_vectors.shape[1]],
    'Sparsity (%)': [
        (1 - bow_matrix.nnz / (bow_matrix.shape[0] * bow_matrix.shape[1])) * 100,
        (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100,
        0  # Dense vector
    ],
    'Memory (MB)': [
        bow_matrix.data.nbytes / 1024**2,
        tfidf_matrix.data.nbytes / 1024**2,
        doc_vectors.nbytes / 1024**2
    ],
    'Thời gian (s)': [time_bow, time_tfidf, time_w2v + time_doc_vec]
}

df_comparison = pd.DataFrame(comparison_data)

print("\nSO SÁNH TỔNG QUAN CÁC PHƯƠNG PHÁP VECTORIZATION:")
print("=" * 100)
print(df_comparison.to_string(index=False))
print("=" * 100)

In [ ]:
# Vẽ biểu đồ so sánh
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

methods = df_comparison['Phương pháp']
colors = ['#3498db', '#e74c3c', '#2ecc71']

# 1. Dimensions
bars = axes[0, 0].bar(methods, df_comparison['Dimensions'], color=colors, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('So sánh số chiều (Dimensions)', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Số chiều', fontsize=12)
axes[0, 0].grid(axis='y', alpha=0.3)
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height):,}', ha='center', va='bottom', fontweight='bold')

# 2. Sparsity
bars = axes[0, 1].bar(methods, df_comparison['Sparsity (%)'], color=colors, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('So sánh độ thưa (Sparsity)', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Sparsity (%)', fontsize=12)
axes[0, 1].grid(axis='y', alpha=0.3)
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

# 3. Memory
bars = axes[1, 0].bar(methods, df_comparison['Memory (MB)'], color=colors, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('So sánh bộ nhớ', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Memory (MB)', fontsize=12)
axes[1, 0].grid(axis='y', alpha=0.3)
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}', ha='center', va='bottom', fontweight='bold')

# 4. Thời gian
bars = axes[1, 1].bar(methods, df_comparison['Thời gian (s)'], color=colors, edgecolor='black', alpha=0.7)
axes[1, 1].set_title('So sánh thời gian xử lý', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Thời gian (giây)', fontsize=12)
axes[1, 1].grid(axis='y', alpha=0.3)
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}s', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Lưu kết quả

In [ ]:
# Lưu DataFrame đã xử lý
output_path = '../data/text/processed_imdb.csv'
df_work[['review', 'sentiment', 'processed_text']].to_csv(output_path, index=False)
print(f"✓ Đã lưu processed data vào: {output_path}")